<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c02-header.png" alt="Nextia Learning · Python for Practical AI Engineering" width="100%">

# Reading and writing files

**[C02-M04-L01 · Reading and writing files](https://learning.nextia-ai.com/courses/python/m04/reading-and-writing-files/)** · C02, Python for Practical AI Engineering · Module 4, lesson 1 of 3 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will learn to** read ticket records from CSV and JSON files with the correct text encoding, separate the valid tickets from the rejected records, and write a JSON report safely.

| | |
|---|---|
| **Time** | About 50 minutes |
| **Needs** | An internet connection for the setup, to download the data files (under 4 KB). No account and no install: only Python. |
| **You should know** | The package with `records.py`, `parsing.py` and `report.py`, from [Structured records](https://learning.nextia-ai.com/courses/python/m03/structured-records/). |
| **You do not need** | Your local project. The setup makes the same `ticket-cleaner` folder here, with the data files. |
| **Tested** | 2026-10-07, Python 3.14.6, in Jupyter on macOS, from a fresh start (*Restart and run all*) |

### How to use this notebook

This notebook has every example, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/python/m04/reading-and-writing-files/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/usmanahmed99/nextia-learning-labs/blob/main/C02/M04-L01-reading-and-writing-files/reading-and-writing-files-solution.ipynb).

## Setup: make the project folder

Run the next cell. It makes a folder `ticket-cleaner`, laid out like your project from Module 1, and works in it. It writes the files that you made in the earlier lessons (the code is in the cell). It needs no internet connection.

It also downloads the course's ticket files (`tickets.csv`, `tickets.json`, under 4 KB together) into `data/`, and checks each file's checksum, as you did with `curl` in [Reading and writing files](https://learning.nextia-ai.com/courses/python/m04/reading-and-writing-files/#get-the-data-file). This part needs an internet connection and no account. If it shows `Could not download`, check your connection and run it again (in Kaggle, turn on **Internet** in the notebook settings). If it shows `wrong checksum`, delete the folder `ticket-cleaner` (in Colab: the **Files** panel on the left), then run it again. The data is made up for this course; it has no real people.

You should see `Ready: ticket-cleaner, 8 files`. If you run the cell again, it puts every file back as at the start of this lesson: use this to start again.

The files are the code of these lessons, as you have it at the end of each one: [Decisions and repetition](https://learning.nextia-ai.com/courses/python/m02/decisions-and-repetition/) (count_tickets.py), [Function contracts](https://learning.nextia-ai.com/courses/python/m03/function-contracts/) (count_tickets.py as functions), [Modules and imports](https://learning.nextia-ai.com/courses/python/m03/modules-and-imports/) (the package `ticket_cleaner` with `report.py`), [Structured records](https://learning.nextia-ai.com/courses/python/m03/structured-records/) (`records.py` and `parsing.py`).

In [ ]:
# Setup: make the ticket-cleaner folder as at the start of this lesson. Download the data files.
import hashlib, os, shutil, subprocess, sys, urllib.request
from pathlib import Path

if Path.cwd().name == "ticket-cleaner":  # the cell ran before: start from the parent folder
    os.chdir("..")
PROJECT = Path("ticket-cleaner").resolve()
for item in PROJECT.glob("*"):  # start again: remove everything except the data files
    if item.name != "data":
        shutil.rmtree(item) if item.is_dir() else item.unlink()
(PROJECT / "data").mkdir(parents=True, exist_ok=True)

LABS = "https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/C02/"
DATA = {
    "tickets.csv": "82e3fb4fa6d94a47b9a7b05d940a088596363e4dd11e89bf82ed486bd2325a1a",
    "tickets.json": "a801f505e3d91ff79cefee9c72e53b5f48e983ee152305b7f04e3ac882dd9946",
}

def download(url, path):
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:
        # Python without certificates (python.org on macOS): try curl.
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

for name, expected in DATA.items():
    path = PROJECT / "data" / name
    if not (path.exists() and hashlib.sha256(path.read_bytes()).hexdigest() == expected):
        download(LABS + "data/" + name, path)
        if hashlib.sha256(path.read_bytes()).hexdigest() != expected:
            raise SystemExit(f"{name} has the wrong checksum. Delete the folder ticket-cleaner and run again.")


FILES = {
    'count_tickets.py': '''\
from ticket_cleaner.parsing import find_problems, to_ticket
from ticket_cleaner.records import Ticket
from ticket_cleaner.report import average_priority, count_by_category, filter_by_status

SAMPLE_ROWS = [
    {"id": "T-1001", "status": "open", "category": "billing", "priority": "2"},
    {"id": "T-1002", "status": "closed", "category": "login", "priority": "3"},
    {"id": "t-1003", "status": "Open ", "category": "login", "priority": "1"},
    {"id": "T-1004", "status": "opne", "category": "billing", "priority": "2"},
    {"id": "T-1005", "status": "open", "category": "billing", "priority": "high"},
    {"id": "T-1006", "status": "closed", "category": "shipping", "priority": "1"},
]


def main() -> None:
    tickets: list[Ticket] = []
    for raw in SAMPLE_ROWS:
        problems = find_problems(raw)
        if problems:
            print(f"Skipped {raw['id']}: {'; '.join(problems)}")
        else:
            tickets.append(to_ticket(raw))

    status = "open"
    selected = filter_by_status(tickets, status)
    print(f"Tickets with status '{status}': {len(selected)}")
    if not selected:
        print("No tickets to count.")
        return
    for category, count in count_by_category(selected).items():
        print(f"  {category}: {count}")
    print(f"Average priority: {average_priority(selected)}")


if __name__ == "__main__":
    main()
''',
    'requirements.txt': '''\
httpx==0.28.1
''',
    'ticket_cleaner/__init__.py': '''\
"""Clean support-ticket records and summarise them."""
''',
    'ticket_cleaner/report.py': '''\
from ticket_cleaner.records import Ticket


def filter_by_status(tickets: list[Ticket], status: str = "open") -> list[Ticket]:
    """Return the tickets that have the given status."""
    return [ticket for ticket in tickets if ticket.status == status]


def count_by_category(tickets: list[Ticket]) -> dict[str, int]:
    """Return the number of tickets in each category, sorted by category."""
    counts = {}
    for ticket in tickets:
        counts[ticket.category] = counts.get(ticket.category, 0) + 1
    return dict(sorted(counts.items()))


def average_priority(tickets: list[Ticket]) -> float | None:
    """Return the average priority, or None when there are no tickets."""
    if not tickets:
        return None
    return round(sum(ticket.priority for ticket in tickets) / len(tickets), 1)
''',
    'ticket_cleaner/records.py': '''\
from dataclasses import dataclass

STATUSES = {"open", "pending", "closed"}
PRIORITIES = {1, 2, 3}
REQUIRED_FIELDS = ("id", "status", "category", "priority")


@dataclass(frozen=True)
class Ticket:
    """One valid support ticket."""

    id: str
    status: str
    category: str
    priority: int

    def is_urgent(self) -> bool:
        """Return True for priority 1, the most urgent level."""
        return self.priority == 1
''',
    'ticket_cleaner/parsing.py': '''\
from ticket_cleaner.records import PRIORITIES, REQUIRED_FIELDS, STATUSES, Ticket


def clean_text(value: object) -> str:
    """Return the value as text with no outer spaces, in small letters."""
    if value is None:
        return ""
    return str(value).strip().lower()


def find_problems(raw: dict) -> list[str]:
    """Return a list of problems with one raw record. An empty list means valid."""
    problems = []
    for field in REQUIRED_FIELDS:
        if clean_text(raw.get(field)) == "":
            problems.append(f"missing {field}")

    status = clean_text(raw.get("status"))
    if status and status not in STATUSES:
        problems.append(f"unknown status {status!r}")

    priority = clean_text(raw.get("priority"))
    if priority and not (priority.isdigit() and int(priority) in PRIORITIES):
        problems.append(f"priority must be 1, 2 or 3, not {priority!r}")
    return problems


def to_ticket(raw: dict) -> Ticket:
    """Make a Ticket from a raw record that has no problems."""
    return Ticket(
        id=clean_text(raw["id"]).upper(),
        status=clean_text(raw["status"]),
        category=clean_text(raw["category"]),
        priority=int(clean_text(raw["priority"])),
    )
''',
}
for name, text in FILES.items():
    (PROJECT / name).parent.mkdir(parents=True, exist_ok=True)
    (PROJECT / name).write_text(text, encoding="utf-8")

os.chdir(PROJECT)  # work in the project folder, as in the lesson
sys.path.insert(0, str(PROJECT))
sys.dont_write_bytecode = True  # always read the newest version of each file
count = sum(1 for path in PROJECT.rglob("*") if path.is_file())
print("Ready: ticket-cleaner" + (f", {count} file{'s' if count > 1 else ''}" if count else ""))

## Setup: the helpers

The next cell defines small functions that this notebook uses. You do not need to read them, but you can.

- `check("""assert ...""")` runs the `assert` lines of a check cell. It prints `Check passed.`, or `Not yet:` and what is different. It does not show the answer.
- `run("count_tickets.py")` runs a file of the project with Python, as `python count_tickets.py` does in your terminal, and shows its output and its exit code. `run("-m", "pytest")` is `python -m pytest`. `env={"NAME": "value"}` sets an environment variable for that one run.
- `fresh("ticket_cleaner.parsing")` imports a module of the project again, so that the notebook uses the newest version of its file. (A plain `import` uses the copy that is already in memory: see [Notebook discipline](https://learning.nextia-ai.com/courses/python/m06/notebook-discipline/).)
- `tree()` shows the files and folders of the project, like the folder trees on the lesson page.
- The check functions of this lesson's tasks. They print `Check passed.` or `Not yet:` and what is different.

Run the cell. It prints nothing.

In [ ]:
# Helpers: run the project, and check an answer without showing it.
import os, sys

def check(asserts):
    """Run the assert lines of a check. Say what is different, never the answer."""
    try:
        exec(asserts, globals())
    except AssertionError as error:
        print(f"Not yet: {error}")
    except NameError as error:
        print(f"Not yet: {error}. Run your cell above first.")
    except Exception as error:
        print(f"Not yet: the check stopped with {type(error).__name__}: {error}")
    else:
        print("Check passed.")

import subprocess, sys

def environment(env=None):
    """The environment variables of a run: the notebook's, plus `env` (None
    removes one). No colours, so the output reads as plain text."""
    variables = {**os.environ, "PYTHONUNBUFFERED": "1", "PYTHONDONTWRITEBYTECODE": "1",
                 "PYTHON_COLORS": "0", "NO_COLOR": "1", "PY_COLORS": "0"}
    variables.pop("FORCE_COLOR", None)
    for name, value in (env or {}).items():
        if value is None:
            variables.pop(name, None)
        else:
            variables[name] = value
    return variables

def run_quiet(*args, env=None):
    """Run `python <args>` in the project folder. Return the result
    (.returncode, .stdout, .stderr)."""
    return subprocess.run([sys.executable, *args], env=environment(env), text=True, capture_output=True)

def run(*args, env=None, streams="both"):
    """Run `python <args>` in the project folder, like the terminal, and show
    the output and the exit code. streams="stdout" or "stderr" shows one only."""
    if streams == "both":  # the two streams together, in the order they were written
        result = subprocess.run([sys.executable, *args], env=environment(env), text=True,
                                stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
        output = result.stdout
    else:
        result = run_quiet(*args, env=env)
        output = result.stdout if streams == "stdout" else result.stderr
    if output:
        print(output, end="" if output.endswith("\n") else "\n")
    print(f"(exit code {result.returncode})")

import importlib

def fresh(module_name):
    """Import a module of the project again, from its newest file."""
    for name in list(sys.modules):
        if name == "ticket_cleaner" or name.startswith("ticket_cleaner."):
            del sys.modules[name]
    importlib.invalidate_caches()
    return importlib.import_module(module_name)

def tree():
    """Show the files and folders of the project, one level of indent per folder."""
    print("ticket-cleaner/")
    for path in sorted(PROJECT.rglob("*")):
        parts = path.relative_to(PROJECT).parts
        if any(part in ("__pycache__", ".pytest_cache") for part in parts):
            continue
        print("    " * len(parts) + path.name + ("/" if path.is_dir() else ""))

import json
from pathlib import Path

def check_pending_report():
    """Check reports/pending.json and the constants of count_tickets.py."""
    source = Path("count_tickets.py").read_text(encoding="utf-8")
    if 'Path("data/tickets.json")' not in source:
        print("Not yet: INPUT_FILE must be data/tickets.json.")
        return
    path = Path("reports/pending.json")
    if not path.exists():
        print("Not yet: there is no reports/pending.json. Change OUTPUT_FILE, then run the cell and the run cell.")
        return
    summary = json.loads(path.read_text(encoding="utf-8"))
    if summary.get("status") != "pending":
        print(f"Not yet: the report is for the status {summary.get('status')!r}. Change the argument of build_summary.")
        return
    if summary.get("valid_records") != 9 or len(summary.get("rejected", [])) != 6:
        print("Not yet: the report must have 9 valid records and 6 rejected ones, as for the CSV file.")
        return
    print("Check passed.")

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. The data file

The setup downloaded the course data into `data/`: 15 synthetic tickets, 9 valid and 6 with one problem each, as `tickets.csv` and as `tickets.json`. On your computer, you download them with `curl`, as on the lesson page.

`Path(...).read_text(encoding="utf-8")` reads a whole text file. Run the cell. It shows the first four lines of the CSV file.

In [ ]:
from pathlib import Path

text = Path("data/tickets.csv").read_text(encoding="utf-8")
for line in text.splitlines()[:4]:
    print(line)

> **Check.** You should see the column names `id,status,category,priority,subject`, then three tickets, from `T-1001,open,billing,2,Charged twice for March`.

## 2. Encodings and with

A file on disk is a series of bytes. To read it as text, Python must know the **encoding**: the rule that turns bytes into letters. The course data uses **UTF-8**. If you do not give an encoding, Python uses the default of the computer, which on Windows is often not UTF-8. So always give it.

The `with` statement opens the file, and **closes it when the block ends**, even if an error stops the block. It is a **context manager**. Run the cell.

In [ ]:
with open("data/tickets.csv", encoding="utf-8") as file:
    text = file.read()

print(len(text), "characters")
print(file.closed)

> **Check.** You should see `758 characters` and `True`: the file is closed after the block.

## 3. Paths with pathlib

A `Path` knows the parts of a path, and joins paths correctly on every system with `/`. A relative path such as `data/tickets.csv` starts from the working directory, so the notebook works in the project folder, as you do in the terminal. Run the cell.

In [ ]:
path = Path("data") / "tickets.csv"
print(path)
print(path.suffix, path.name, path.parent)
print(path.exists())
print(Path("reports/summary.json").exists())

> **Check.** You should see `data/tickets.csv`, `.csv tickets.csv data`, `True` and `False`: there is no report yet.

## 4. CSV and JSON

`csv.DictReader` gives each row of a CSV file as a dictionary, with the column names as keys. `json.load` turns JSON text into Python values.

> **Predict.** Is the priority of the first ticket the same type in both files? Then run the cell.

In [ ]:
import csv
import json

with open("data/tickets.csv", encoding="utf-8", newline="") as file:
    csv_rows = list(csv.DictReader(file))
print(csv_rows[0])

with open("data/tickets.json", encoding="utf-8") as file:
    json_rows = json.load(file)
print(json_rows[0])

> **Check.** You should see the same ticket twice, but the priority is `'2'` (a string) from the CSV file and `2` (a number) from the JSON file. A CSV file has no types. `find_problems` accepts both, because `clean_text` turns any value into text first. `newline=""` lets the `csv` module handle line endings itself.

`"w"` opens a file for writing, and makes it new. `\n` is a line break. Run the cell. It writes a small practice file, reads it, and deletes it.

In [ ]:
with open("practice.csv", "w", encoding="utf-8") as file:
    file.write("id,status,priority\n")
    file.write("T-1001,open,2\n")
    file.write("T-1002,Closed ,3\n")

with open("practice.csv", encoding="utf-8", newline="") as file:
    rows = list(csv.DictReader(file))

print(len(rows), "rows")
for row in rows:
    print(row)
print("Type of priority:", type(rows[0]["priority"]))
Path("practice.csv").unlink()

> **Check.** You should see `2 rows`, the two rows as dictionaries, and `Type of priority: <class 'str'>`.

> **Your turn: change one thing.** Add a third row, `T-1003,pending,1`, and run the cell again. You should see `3 rows`.

## 5. Worked example: read, split, report, write

The program has four steps. Each one is a function with one job. Run the next five cells. Read the lesson page for an explanation of each one.

**1. Read the rows.** A new module, `files.py`, reads a CSV or a JSON file into a list of dictionaries. `raise` stops the function with an error for any other file type. It also has `write_json` (step 4).

In [ ]:
%%writefile ticket_cleaner/files.py
import csv
import json
import os
from pathlib import Path


def read_rows(path: Path) -> list[dict]:
    """Read raw records from a .csv or .json file."""
    if path.suffix == ".csv":
        with open(path, encoding="utf-8", newline="") as file:
            return list(csv.DictReader(file))
    if path.suffix == ".json":
        with open(path, encoding="utf-8") as file:
            return json.load(file)
    raise ValueError(f"cannot read {path.name}: use a .csv or .json file")


def write_json(data: dict, path: Path) -> None:
    """Save data as JSON. Write a temporary file first, so a failure never leaves half a file."""
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + ".tmp")
    with open(temporary, "w", encoding="utf-8") as file:
        json.dump(data, file, indent=2, ensure_ascii=False)
        file.write("\n")
    os.replace(temporary, path)

**2. Split the valid tickets from the rejected records.** A rejected record gets its own small type, `Rejected`, with the row number and the problems. `split_records` also finds a ticket ID that appears twice. `enumerate(rows, start=1)` gives each row with its number. Every input row ends in exactly one of the two lists: nothing is dropped silently.

In [ ]:
%%writefile ticket_cleaner/records.py
from dataclasses import dataclass

STATUSES = {"open", "pending", "closed"}
PRIORITIES = {1, 2, 3}
REQUIRED_FIELDS = ("id", "status", "category", "priority")


@dataclass(frozen=True)
class Ticket:
    """One valid support ticket."""

    id: str
    status: str
    category: str
    priority: int

    def is_urgent(self) -> bool:
        """Return True for priority 1, the most urgent level."""
        return self.priority == 1


@dataclass(frozen=True)
class Rejected:
    """An input record that is not a valid ticket, and why."""

    row: int
    problems: list[str]

In [ ]:
%%writefile ticket_cleaner/parsing.py
from ticket_cleaner.records import PRIORITIES, REQUIRED_FIELDS, STATUSES, Rejected, Ticket


def clean_text(value: object) -> str:
    """Return the value as text with no outer spaces, in small letters."""
    if value is None:
        return ""
    return str(value).strip().lower()


def find_problems(raw: dict) -> list[str]:
    """Return a list of problems with one raw record. An empty list means valid."""
    problems = []
    for field in REQUIRED_FIELDS:
        if clean_text(raw.get(field)) == "":
            problems.append(f"missing {field}")

    status = clean_text(raw.get("status"))
    if status and status not in STATUSES:
        problems.append(f"unknown status {status!r}")

    priority = clean_text(raw.get("priority"))
    if priority and not (priority.isdigit() and int(priority) in PRIORITIES):
        problems.append(f"priority must be 1, 2 or 3, not {priority!r}")
    return problems


def to_ticket(raw: dict) -> Ticket:
    """Make a Ticket from a raw record that has no problems."""
    return Ticket(
        id=clean_text(raw["id"]).upper(),
        status=clean_text(raw["status"]),
        category=clean_text(raw["category"]),
        priority=int(clean_text(raw["priority"])),
    )


def split_records(rows: list[dict]) -> tuple[list[Ticket], list[Rejected]]:
    """Separate valid tickets from rejected records. Nothing is dropped silently."""
    tickets = []
    rejected = []
    seen_ids = set()
    for row_number, raw in enumerate(rows, start=1):
        problems = find_problems(raw)
        if not problems:
            ticket = to_ticket(raw)
            if ticket.id in seen_ids:
                problems.append(f"duplicate id {ticket.id}")
        if problems:
            rejected.append(Rejected(row=row_number, problems=problems))
        else:
            tickets.append(ticket)
            seen_ids.add(ticket.id)
    return tickets, rejected

**3. Build the report.** `build_summary` returns a dictionary with only types that JSON can store. **4. Write the report safely:** `write_json` (in `files.py`) writes a temporary file first, then renames it with `os.replace` in one step. So a reader sees the old complete file or the new complete file, never half a file.

In [ ]:
%%writefile ticket_cleaner/report.py
from ticket_cleaner.records import Rejected, Ticket


def filter_by_status(tickets: list[Ticket], status: str = "open") -> list[Ticket]:
    """Return the tickets that have the given status."""
    return [ticket for ticket in tickets if ticket.status == status]


def count_by_category(tickets: list[Ticket]) -> dict[str, int]:
    """Return the number of tickets in each category, sorted by category."""
    counts = {}
    for ticket in tickets:
        counts[ticket.category] = counts.get(ticket.category, 0) + 1
    return dict(sorted(counts.items()))


def average_priority(tickets: list[Ticket]) -> float | None:
    """Return the average priority, or None when there are no tickets."""
    if not tickets:
        return None
    return round(sum(ticket.priority for ticket in tickets) / len(tickets), 1)


def build_summary(tickets: list[Ticket], rejected: list[Rejected], status: str) -> dict:
    """Return the report as a dictionary that can be saved as JSON."""
    selected = filter_by_status(tickets, status)
    return {
        "status": status,
        "valid_records": len(tickets),
        "selected": len(selected),
        "by_category": count_by_category(selected),
        "average_priority": average_priority(selected),
        "rejected": [{"row": r.row, "problems": r.problems} for r in rejected],
    }

**The script** now reads the file instead of the sample rows.

In [ ]:
%%writefile count_tickets.py
from pathlib import Path

from ticket_cleaner.files import read_rows, write_json
from ticket_cleaner.parsing import split_records
from ticket_cleaner.report import build_summary

INPUT_FILE = Path("data/tickets.csv")
OUTPUT_FILE = Path("reports/summary.json")


def main() -> None:
    rows = read_rows(INPUT_FILE)
    tickets, rejected = split_records(rows)
    for record in rejected:
        print(f"Row {record.row} rejected: {'; '.join(record.problems)}")

    summary = build_summary(tickets, rejected, status="open")
    write_json(summary, OUTPUT_FILE)
    print(f"{len(rows)} rows: {len(tickets)} valid, {len(rejected)} rejected.")
    print(f"Report: {OUTPUT_FILE}")


if __name__ == "__main__":
    main()

## 6. Guided practice: the first real report

Run the script.

In [ ]:
run("count_tickets.py")

> **Check.** You should see six lines `Row … rejected: …` (rows 7, 8, 9, 11, 13 and 14), then `15 rows: 9 valid, 6 rejected.` and `Report: reports/summary.json`.

Read the report, and check that nothing is lost: valid plus rejected must be all 15 rows. Run the cell.

In [ ]:
summary = json.loads(Path("reports/summary.json").read_text(encoding="utf-8"))
print("selected:", summary["selected"])
print("rejected:", len(summary["rejected"]))
print("valid + rejected:", summary["valid_records"] + len(summary["rejected"]))
print(summary["rejected"][0])
tree()

> **Check.** You should see `selected: 4`, `rejected: 6`, `valid + rejected: 15` and `{'row': 7, 'problems': ["unknown status 'opne'"]}`. The tree shows the new folder `reports/` with `summary.json`, and no `.tmp` file. Row 7 is on line 8 of the CSV file, because line 1 has the column names.

## 7. Your turn: JSON input, a second report

> **Your turn.** In the next cell, change the script so that it reads `data/tickets.json`, and writes the report for the **pending** tickets to `reports/pending.json`. Change only the two constants and one argument. Predict the numbers before you run it. Run the cell, the `run` cell, then the check cell.

In [ ]:
%%writefile count_tickets.py
from pathlib import Path

from ticket_cleaner.files import read_rows, write_json
from ticket_cleaner.parsing import split_records
from ticket_cleaner.report import build_summary

INPUT_FILE = Path("data/tickets.csv")
OUTPUT_FILE = Path("reports/summary.json")


def main() -> None:
    rows = read_rows(INPUT_FILE)
    tickets, rejected = split_records(rows)
    for record in rejected:
        print(f"Row {record.row} rejected: {'; '.join(record.problems)}")

    summary = build_summary(tickets, rejected, status="open")
    write_json(summary, OUTPUT_FILE)
    print(f"{len(rows)} rows: {len(tickets)} valid, {len(rejected)} rejected.")
    print(f"Report: {OUTPUT_FILE}")


if __name__ == "__main__":
    main()

In [ ]:
run("count_tickets.py")

In [ ]:
check_pending_report()

Then look at the new report. `reports/summary.json` did not change: each run writes only its own output file. Run the cell.

In [ ]:
for name in ["pending.json", "summary.json"]:
    path = Path("reports") / name
    if path.exists():
        report = json.loads(path.read_text(encoding="utf-8"))
        print(name, report["status"], report["selected"], report["by_category"], report["average_priority"])

> **Check.** When your change is correct, you should see `pending.json pending 2 {'billing': 1, 'login': 1} 1.5` and `summary.json open 4 …`. Row 10 (`LOGIN`) counts as `login`, because `to_ticket` cleaned it.

## 8. Failure case: strange letters such as MÃ¼ller

Python reads a file with a different encoding than the one that it was saved in. The file is UTF-8, where `ü` is two bytes. An older encoding, such as Windows' `cp1252`, reads those two bytes as two letters.

> **Predict.** What does row 12 look like when it is read with the wrong encoding? Then run the cell.

In [ ]:
raw_bytes = Path("data/tickets.csv").read_bytes()
print(raw_bytes.decode("utf-8").splitlines()[12])
print(raw_bytes.decode("cp1252").splitlines()[12])

> **Check.** You should see `Change the company name to Müller GmbH` at the end of the first line, and `MÃ¼ller GmbH` at the end of the second. There is no error: the report just gets wrong letters.

**Fix:** every `open(` in your code must have `encoding="utf-8"`. If a report was written with wrong letters, run the program again to write it correctly.

> **Warning.** `FileNotFoundError: [Errno 2] No such file or directory: 'data/tickets.csv'` means that the path does not lead to the file from the working directory. Check that the terminal is in `ticket-cleaner`, and that the file name is exactly `tickets.csv`.

## Check your understanding and recap

Answer the **knowledge checks** on the [lesson page](https://learning.nextia-ai.com/courses/python/m04/reading-and-writing-files/). They count toward your certificate when you are signed in and enrolled. On your computer, download the data with `curl` and build the same files in your project, as in the guided practice of the lesson.

In this lesson you learned to open text files with `encoding="utf-8"` inside a `with` block, to read CSV and JSON into a list of dictionaries, to split the input into valid and rejected records with a reason for each, and to write a report through a temporary file.

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Basic HTTP client use](https://learning.nextia-ai.com/courses/python/m04/basic-http-client-use/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/python/m04/reading-and-writing-files/)

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The tickets in this course are made up; they have no real people.